# 01 Data Exploration: LinkedIn Job Postings

Objective: verify the Kaggle LinkedIn Job Postings dataset layout, inspect the raw fields needed by ResuMatch, and document preprocessing choices for embeddings, retrieval, clustering, and salary modeling.

This notebook is intentionally safe to run before the raw Kaggle data is downloaded. If `data/raw/` is empty, the first cells print setup instructions and the later cells return empty summaries instead of failing.

## Setup

The expected source is Kaggle dataset `arshkon/linkedin-job-postings`. Raw files should be unzipped under `data/raw/`; generated outputs from `scripts/preprocess_data.py` should land under `data/processed/`.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:  # Allows lightweight script execution outside Jupyter.
    def display(obj):
        print(obj)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_JOBS = PROJECT_ROOT / "data" / "processed" / "jobs.parquet"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.preprocess_data import (
    FILE_ALIASES,
    annualize_salaries,
    clean_skills,
    discover_input_files,
    make_embedding_text,
    map_experience_level,
    normalize_columns,
    strip_html,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)
SEED = 42
np.random.seed(SEED)

## Raw Data Availability

The preprocessing script searches recursively, so the files can be in Kaggle's default nested folders. The required inputs are `postings.csv` or `job_postings.csv`, plus `companies.csv`.

In [ ]:
def find_csvs(raw_dir: Path) -> pd.DataFrame:
    rows = []
    for path in sorted(raw_dir.rglob("*.csv")) if raw_dir.exists() else []:
        rows.append({"relative_path": path.relative_to(PROJECT_ROOT).as_posix(), "size_mb": path.stat().st_size / 1_000_000})
    return pd.DataFrame(rows)

csv_inventory = find_csvs(RAW_DIR)
if csv_inventory.empty:
    print("No raw Kaggle CSVs found under data/raw/.")
    print("Download manually from Kaggle or run:")
    print("kaggle datasets download -d arshkon/linkedin-job-postings -p data/raw --unzip")
else:
    display(csv_inventory)

try:
    discovered_paths = discover_input_files(RAW_DIR)
except FileNotFoundError as exc:
    discovered_paths = {key: None for key in FILE_ALIASES}
    print(f"Discovery status: {exc}")

discovery_summary = pd.DataFrame(
    [
        {"logical_input": key, "path": None if path is None else path.relative_to(PROJECT_ROOT).as_posix()}
        for key, path in discovered_paths.items()
    ]
)
display(discovery_summary)

## Load Raw Tables

Only small previews are displayed. Full raw files can be large, so avoid printing entire frames.

In [ ]:
def read_csv_if_present(path: Path | None, nrows: int | None = None) -> pd.DataFrame:
    if path is None:
        return pd.DataFrame()
    return normalize_columns(pd.read_csv(path, nrows=nrows))

raw_jobs = read_csv_if_present(discovered_paths.get("job_postings"))
raw_companies = read_csv_if_present(discovered_paths.get("companies"))
raw_benefits = read_csv_if_present(discovered_paths.get("benefits"))
raw_employee_counts = read_csv_if_present(discovered_paths.get("employee_counts"))

raw_tables = {
    "postings": raw_jobs,
    "companies": raw_companies,
    "benefits": raw_benefits,
    "employee_counts": raw_employee_counts,
}

summary_rows = []
for name, frame in raw_tables.items():
    summary_rows.append({"table": name, "rows": len(frame), "columns": len(frame.columns)})
summary = pd.DataFrame(summary_rows)
display(summary)

if not raw_jobs.empty:
    display(raw_jobs.head(3))

## Schema And Missingness

These checks identify which raw columns are reliable enough for downstream ML, and which fields need fallbacks.

In [ ]:
KEY_COLUMNS = [
    "job_id", "company_id", "company_name", "title", "description", "skills_desc",
    "min_salary", "med_salary", "max_salary", "pay_period", "normalized_salary",
    "currency", "location", "formatted_experience_level", "formatted_work_type",
    "work_type", "remote_allowed",
]

if raw_jobs.empty:
    print("Raw postings table unavailable; schema/missingness skipped.")
else:
    available = [column for column in KEY_COLUMNS if column in raw_jobs.columns]
    missing = [column for column in KEY_COLUMNS if column not in raw_jobs.columns]
    print(f"Available key columns ({len(available)}): {available}")
    print(f"Missing key columns ({len(missing)}): {missing}")

    missingness = (
        raw_jobs[available]
        .isna()
        .mean()
        .sort_values(ascending=False)
        .rename("null_rate")
        .reset_index()
        .rename(columns={"index": "column"})
    )
    display(missingness)

## Salary And Pay Periods

ResuMatch predicts salary distributions, so we need annualized salary targets. The preprocessing script uses `med_salary`, falling back to midpoint/min/max, then multiplies by pay period.

In [ ]:
if raw_jobs.empty:
    print("Raw postings table unavailable; salary analysis skipped.")
else:
    salary_cols = [column for column in ["min_salary", "med_salary", "max_salary", "pay_period", "normalized_salary", "currency"] if column in raw_jobs.columns]
    display(raw_jobs[salary_cols].head(10))

    salary_annual = annualize_salaries(raw_jobs)
    salary_summary = salary_annual.dropna().describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).to_frame("salary_annual")
    print(f"Rows with annualized salary: {salary_annual.notna().sum():,} / {len(raw_jobs):,}")
    display(salary_summary)

    if "pay_period" in raw_jobs.columns:
        pay_periods = raw_jobs["pay_period"].fillna("<missing>").astype(str).str.lower().value_counts().head(20).rename_axis("pay_period").reset_index(name="rows")
        display(pay_periods)

## Titles, Companies, Locations, And Work Types

These fields drive app filters, user-facing match cards, and basic dataset sanity checks.

In [ ]:
def top_counts(frame: pd.DataFrame, column: str, n: int = 15) -> pd.DataFrame:
    if frame.empty or column not in frame.columns:
        return pd.DataFrame(columns=[column, "rows"])
    return (
        frame[column]
        .fillna("<missing>")
        .astype(str)
        .str.strip()
        .replace("", "<blank>")
        .value_counts()
        .head(n)
        .rename_axis(column)
        .reset_index(name="rows")
    )

if raw_jobs.empty:
    print("Raw postings table unavailable; categorical summaries skipped.")
else:
    for column in ["title", "company_name", "location", "formatted_experience_level", "formatted_work_type", "work_type"]:
        print(f"Top {column}")
        display(top_counts(raw_jobs, column, n=10))

## Text Fields For Embeddings

The embedding text should combine role title, cleaned description, and deduplicated skills. This is the text that `ml.embeddings.Encoder` will encode for FAISS retrieval and clustering.

In [ ]:
if raw_jobs.empty:
    print("Raw postings table unavailable; text analysis skipped.")
else:
    sample = raw_jobs.head(5).copy()
    description = sample["description"] if "description" in sample.columns else pd.Series("", index=sample.index)
    skills = sample["skills_desc"] if "skills_desc" in sample.columns else pd.Series("", index=sample.index)
    title = sample["title"] if "title" in sample.columns else pd.Series("", index=sample.index)
    sample["description_clean"] = description.map(strip_html)
    sample["skills_desc_clean"] = skills.map(clean_skills)
    sample["embedding_text"] = [
        make_embedding_text(t, d, s)
        for t, d, s in zip(title, description, skills, strict=True)
    ]
    display(sample[[col for col in ["job_id", "title", "description_clean", "skills_desc_clean", "embedding_text"] if col in sample.columns]])

    text_source = [
        make_embedding_text(t, d, s)
        for t, d, s in zip(title if len(title) == len(raw_jobs) else raw_jobs.get("title", pd.Series("", index=raw_jobs.index)),
                           raw_jobs.get("description", pd.Series("", index=raw_jobs.index)),
                           raw_jobs.get("skills_desc", pd.Series("", index=raw_jobs.index)),
                           strict=True)
    ]
    text_lengths = pd.Series([len(text) for text in text_source], name="embedding_text_length")
    display(text_lengths.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).to_frame())

## Processed Data Check

If preprocessing has already run, inspect the generated parquet that downstream embedding/indexing code uses.

In [ ]:
if not PROCESSED_JOBS.exists():
    print("Processed jobs parquet is missing.")
    print("Run: uv run python scripts/preprocess_data.py")
else:
    processed = pd.read_parquet(PROCESSED_JOBS)
    print(f"Processed shape: {processed.shape}")
    display(processed.head(5))
    required_processed = ["job_id", "title", "company_name", "salary_annual", "location", "experience_level", "text"]
    processed_missing = [column for column in required_processed if column not in processed.columns]
    print(f"Missing required processed columns: {processed_missing}")
    display(processed[required_processed].isna().mean().rename("null_rate").reset_index().rename(columns={"index": "column"}))

## Preprocessing Conclusions

- Required raw files: `postings.csv` or `job_postings.csv`, plus `companies.csv`.
- Optional enrichment files: `jobs/benefits.csv` and `companies/employee_counts.csv`.
- Salary target: use `med_salary`, fallback to midpoint/min/max, then annualize by `pay_period`.
- Embedding text: lowercase `title + cleaned description + deduplicated skills_desc`.
- App/retrieval metadata should keep `job_id`, `title`, `company_name`, `salary_annual`, `location`, and `experience_level`.
- Current blocker for real embedding/index generation is raw Kaggle data, not code structure.

Next command after raw data is present:

```bash
uv run python scripts/preprocess_data.py
uv run python scripts/build_index.py
```